# 🧪 Sesión 10 · Hito 1: Mock API RESTful con OpenAPI y Validation Harness

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Tu API ya se comporta como un servicio profesional?

---

## 🖥️ Así se verá el resultado final

```http
POST /api/libros              -> 201 Created
GET  /api/libros              -> 200 OK
GET  /api/libros/1            -> 200 OK
PUT  /api/libros/1            -> 200 OK
DELETE /api/libros/1          -> 204 No Content
GET  /api/libros/1            -> 404 Not Found
POST /api/libros (inválido)   -> 422 Unprocessable Entity
PATCH /api/libros/1           -> 405 Method Not Allowed
GET  /openapi.json            -> 200 OK

RESUMEN: 9/9 casos correctos. ¡Hito 1 listo!
```

_La suite completa es el resultado: si un caso falla, el hito aún no está listo._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Integrar los conceptos del bloque en una Mock API coherente.
- Verificar el contrato completo con una suite de casos.
- Etiquetar la entrega del primer hito en GitHub.

## ✅ Criterios de aceptación

El verificador ejecutará estos casos. Cada uno debe quedar en verde:

- [ ] Listado inicial vacío → HTTP 200
- [ ] Creación válida → HTTP 201
- [ ] Consulta del libro creado → HTTP 200
- [ ] Actualización del libro → HTTP 200
- [ ] Eliminación del libro → HTTP 204
- [ ] El libro eliminado ya no existe → HTTP 404
- [ ] Validación de datos inválidos → HTTP 422
- [ ] Método no permitido → HTTP 405
- [ ] Contrato OpenAPI disponible → HTTP 200

## 🧩 El reto

Integra una Mock API de libros que cumpla el contrato completo del Bloque 1: CRUD, validación, documentación y errores semánticos.

**Requisitos del contrato**

- CRUD completo de /api/libros con ids asignados por el servidor.
- Validación de titulo y anio con 422 cuando corresponda.
- OpenAPI disponible en /openapi.json con el esquema Libro.
- Errores 404 y 405 semánticos, sin 500.
- Los nueve casos de la suite deben quedar en verde.

> **Pista conceptual:** No empieces por el código: empieza por la tabla de casos. Cada fila es una promesa que tu API debe cumplir.

## 📦 Entregable

Cuaderno en verde, bitácora `auditorias/auditoria_sesion10.md` y tag `v0.1-hito1` en tu repositorio.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [4]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [8]:
%%writefile app.py
import time
import uuid
from fastapi import FastAPI, Request, Response
from fastapi.responses import JSONResponse
from fastapi.exceptions import RequestValidationError
from pydantic import BaseModel, Field

app = FastAPI(
    title="Biblioteca API",
    version="1.0.0",
    description="API de libros del curso DAB-2201"
)

# --- Middleware Sesión 09 ---
@app.middleware("http")
async def middleware_tiempo_y_traza(request: Request, call_next):
    inicio = time.time()
    request_id = request.headers.get("X-Request-ID") or str(uuid.uuid4())
    response = await call_next(request)
    response.headers["X-Process-Time"] = str(time.time() - inicio)
    response.headers["X-Request-ID"] = request_id
    return response

# --- Errores Sesión 08 ---
class LibroNoEncontrado(Exception):
    def __init__(self, libro_id: int):
        self.libro_id = libro_id

@app.exception_handler(LibroNoEncontrado)
async def handler_no_encontrado(request: Request, exc: LibroNoEncontrado):
    return JSONResponse(
        status_code=404,
        content={"error": {"codigo": "NO_ENCONTRADO", "mensaje": f"El libro {exc.libro_id} no existe"}}
    )

@app.exception_handler(RequestValidationError)
async def handler_validacion(request: Request, exc: RequestValidationError):
    return JSONResponse(
        status_code=422,
        content={"error": {"codigo": "VALIDACION", "mensaje": "Revisa los datos enviados"}}
    )

# --- Modelos Sesión 06 y 07 ---
# OJO: Tiene que llamarse Libro para que salga en components.schemas
class Libro(BaseModel):
    id: int
    titulo: str
    anio: int
    precio: float

class LibroCreate(BaseModel):
    titulo: str = Field(min_length=3, max_length=80)
    anio: int = Field(default=1963, ge=1450, le=2100)
    precio: float = Field(default=250.5, gt=0)

class LibroUpdate(BaseModel):
    titulo: str | None = Field(default=None, min_length=3, max_length=80)
    anio: int | None = Field(default=None, ge=1450, le=2100)
    precio: float | None = Field(default=None, gt=0)

libros_db = []
contador = 1

@app.get("/api/libros", summary="Listar libros")
def listar_libros():
    return {"libros": libros_db}

@app.post("/api/libros", status_code=201, summary="Crear un libro", response_model=Libro)
def crear_libro(libro: LibroCreate):
    global contador
    nuevo = {"id": contador, "titulo": libro.titulo, "anio": libro.anio, "precio": libro.precio}
    libros_db.append(nuevo)
    contador += 1
    return nuevo

@app.get("/api/libros/{libro_id}", summary="Obtener un libro", response_model=Libro)
def obtener_libro(libro_id: int):
    for l in libros_db:
        if l["id"] == libro_id:
            return l
    raise LibroNoEncontrado(libro_id)

@app.put("/api/libros/{libro_id}", summary="Actualizar un libro", response_model=Libro)
def actualizar_libro(libro_id: int, datos: LibroUpdate):
    for l in libros_db:
        if l["id"] == libro_id:
            if datos.titulo is not None:
                l["titulo"] = datos.titulo
            if datos.anio is not None:
                l["anio"] = datos.anio
            if datos.precio is not None:
                l["precio"] = datos.precio
            return l
    raise LibroNoEncontrado(libro_id)

@app.delete("/api/libros/{libro_id}", status_code=204, summary="Eliminar un libro")
def eliminar_libro(libro_id: int):
    for i, l in enumerate(libros_db):
        if l["id"] == libro_id:
            del libros_db[i]
            return Response(status_code=204)
    raise LibroNoEncontrado(libro_id)

Overwriting app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [9]:
from verificar import probar_http

casos = [{'descripcion': 'Listado inicial vacío',
  'metodo': 'GET',
  'ruta': '/api/libros',
  'estado': 200,
  'json_contiene': {'libros': []}},
 {'descripcion': 'Creación válida',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'json': {'titulo': 'Rayuela', 'anio': 1963},
  'estado': 201,
  'json_contiene': {'id': 1, 'titulo': 'Rayuela'}},
 {'descripcion': 'Consulta del libro creado',
  'metodo': 'GET',
  'ruta': '/api/libros/1',
  'estado': 200,
  'json_contiene': {'id': 1, 'titulo': 'Rayuela'}},
 {'descripcion': 'Actualización del libro',
  'metodo': 'PUT',
  'ruta': '/api/libros/1',
  'json': {'titulo': 'Rayuela (edición crítica)', 'anio': 1963},
  'estado': 200,
  'json_contiene': {'id': 1, 'titulo': 'Rayuela (edición crítica)'}},
 {'descripcion': 'Eliminación del libro', 'metodo': 'DELETE', 'ruta': '/api/libros/1', 'estado': 204},
 {'descripcion': 'El libro eliminado ya no existe', 'metodo': 'GET', 'ruta': '/api/libros/1', 'estado': 404},
 {'descripcion': 'Validación de datos inválidos',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'json': {'titulo': '', 'anio': 1200},
  'estado': 422},
 {'descripcion': 'Método no permitido', 'metodo': 'PATCH', 'ruta': '/api/libros/1', 'estado': 405},
 {'descripcion': 'Contrato OpenAPI disponible',
  'metodo': 'GET',
  'ruta': '/openapi.json',
  'estado': 200,
  'json_contiene': {'paths': {'/api/libros': {}}, 'components': {'schemas': {'Libro': {}}}}}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 9 casos · app.py
✅ 01. Listado inicial vacío
✅ 02. Creación válida
✅ 03. Consulta del libro creado
✅ 04. Actualización del libro
✅ 05. Eliminación del libro
✅ 06. El libro eliminado ya no existe
✅ 07. Validación de datos inválidos
✅ 08. Método no permitido
✅ 09. Contrato OpenAPI disponible
------------------------------------------------------------------------------
RESULTADO: 9/9 casos correctos. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion10.md` a `auditorias/auditoria_sesion10.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
